# Глава 7. Тонкая настройка по инструкциям

In [1]:
from importlib.metadata import version

pkgs = [
    "numpy",       # Зависимость для PyTorch и TensorFlow
    "matplotlib",  # Библиотека для построения графиков
    "tiktoken",    # Токенизатор
    "torch",       # Библиотека для глубокого обучения
    "tqdm",        # Прогресс-бар
    "tensorflow",  # Для предобученных весов OpenAI
]
for p in pkgs:
    print(f"{p} версия: {version(p)}")

numpy версия: 2.4.6
matplotlib версия: 3.10.9
tiktoken версия: 0.13.0
torch версия: 2.12.0
tqdm версия: 4.67.3
tensorflow версия: 2.21.0


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/01.webp" width=800px>

&nbsp;
## 7.1. Введение в тонкую настройку по инструкциям

- Предобучение LLM включает в себя процедуру обучения, в ходе которой она учится генерировать по одному слову за раз
- Следовательно, предобученная LLM хорошо справляется с дополнением текста, но плохо следует инструкциям
- В этой главе мы научим LLM лучше следовать инструкциям

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/02.webp" width=800px>

&nbsp;
## 7.2. Подготовка набора данных для контролируемой тонкой настройки по инструкции

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/03.webp" width=800px>

- Мы будем работать с набором инструктивных данных

In [1]:
import json
import os
import requests


def download_and_load_file(file_path, url):
    if not os.path.exists(file_path):
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        text_data = response.text
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data


# Изначально использовался код ниже.
# Однако urllib использует старые настройки протокола,
# что может вызывать проблемы у некоторых читателей, использующих VPN.
# Версия с `requests` более надёжна в этом отношении.

"""
import urllib

def download_and_load_file(file_path, url):

    if not os.path.exists(file_path):
        with urllib.request.urlopen(url) as response:
            text_data = response.read().decode("utf-8")
        with open(file_path, "w", encoding="utf-8") as file:
            file.write(text_data)

    else:
        with open(file_path, "r", encoding="utf-8") as file:
            text_data = file.read()

    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    return data
"""


file_path = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)

data = download_and_load_file(file_path, url)
print("Количество записей:", len(data))

Количество записей: 1100


- Каждый элемент в списке `data`, который мы загрузили из JSON-файла выше, представляет собой словарь следующего вида

In [2]:
print("Пример записи:\n", data[50])

Пример записи:
 {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}


- Обратите внимание, что поле `'input'` может быть пустым:

In [3]:
print("Ещё один пример записи:\n", data[999])

Ещё один пример записи:
 {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}


- Тонкую настройку по инструкции часто называют «инструктивной тонкой настройкой с учителем», поскольку она включает обучение модели на наборе данных, где пары «вход-выход» явно заданы
- Существуют различные способы форматирования записей в качестве входных данных для LLM; на рисунке ниже показаны два примера форматов, которые использовались для обучения LLM Alpaca (https://crfm.stanford.edu/2023/03/13/alpaca.html) и Phi-3 (https://arxiv.org/abs/2404.14219) соответственно

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch07_compressed/04.webp?2" width=800px>

 ---

 ## Упражнение 7.1. Изменение стилей подсказок

Предположим, у нас есть следующая запись данных:

```json
{
  "instruction": "Identify the correct spelling of the following word.",
  "input": "Ocassion",
  "output": "The correct spelling is 'Occasion.'"
}
```

Мы форматировали её в соответствии с шаблоном промптов в стиле Alpaca:

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Occassion

### Response:
The correct spelling is 'Occasion.'
```

В этом упражнении мы теперь используем шаблон промптов Phi-3, который форматирует запись данных следующим образом:

```
<user>
Identify the correct spelling of the following word: 'Occasion'

<assistant>
The correct spelling is 'Occasion'.
```

Обратите внимание, что этот шаблон промптов существенно короче, что снижает требования к времени выполнения и аппаратному обеспечению для тонкой настройки LLM и генерации текста, поскольку входные промпты короче.
Чтобы внести это изменение, мы обновляем функцию `format_input` следующим образом:

In [4]:
def format_input(entry):
    instruction_text = (
        f"<|user|>\n{entry['instruction']}"
    )

    input_text = f"\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

In [5]:
sample_data = [
    {'instruction': 'Identify the correct spelling of the following word.', 'input': 'Ocassion', 'output': "The correct spelling is 'Occasion.'"}, 
    {'instruction': "What is an antonym of 'complicated'?", 'input': '', 'output': "An antonym of 'complicated' is 'simple'."}
]

print(format_input(sample_data[0]))
print()
print(format_input(sample_data[1]))

<|user|>
Identify the correct spelling of the following word.
Ocassion

<|user|>
What is an antonym of 'complicated'?


Далее мы также обновим класс `InstructionDataset`, чтобы он использовал шаблон промпта `<|assistant|>` для ответа:

Давайте убедимся, что это работает как задумано, применив её к двум входным образцам — одному с содержимым в поле `'input'` и одному без:

In [6]:
import tiktoken
from torch.utils.data import Dataset

class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data

        # Предварительно токенизируем тексты
        self.encoded_texts = []
        for entry in data:

            ###################################################################
            # НОВОЕ: Используем `format_input_phi` и изменяем шаблон текста ответа
            instruction_plus_input = format_input(entry)
            response_text = f"\n<|assistant|>:\n{entry['output']}"
            ###################################################################
            full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
                tokenizer.encode(full_text)
            )

    def __getitem__(self, index):
        return self.encoded_texts[index]

    def __len__(self):
        return len(self.data)


tokenizer = tiktoken.get_encoding("gpt2")

Наконец, мы также должны обновить способ извлечения сгенерированного ответа при сборе ответов тестового набора:

```python
for i, entry in tqdm(enumerate(test_data), total=len(test_data)):

    input_text = format_input(entry)
    tokenizer=tokenizer

    token_ids = generate(
        model=model,
        idx=text_to_token_ids(input_text, tokenizer).to(device),
        max_new_tokens=256,
        context_size=BASE_CONFIG["context_length"],
        eos_id=50256
    )
    generated_text = token_ids_to_text(token_ids, tokenizer)

    # Новое: Заменяем ###Response на <|assistant|>
    response_text = generated_text[len(input_text):].replace("<|assistant|>:", "").strip()

    test_data[i]["model_response"] = response_text
```

Для вашего удобства решение упражнения реализовано в скрипте [exercise_experiments.py](exercise_experiments.py):

```bash
python exercise_experiments.py --exercise_solution phi3_prompt
```

Вывод:

```
matplotlib версия: 3.7.1
tiktoken версия: 0.7.0
torch версия: 2.3.0+cu121
tqdm версия: 4.66.4
tensorflow версия: 2.15.0
--------------------------------------------------
Длина обучающего набора: 935
Длина валидационного набора: 55
Длина тестового набора: 110
--------------------------------------------------
Устройство: cuda
--------------------------------------------------
...
Загружена модель: gpt2-medium (355M)
--------------------------------------------------
Начальные потери
   Потери на обучении: 3.71630220413208
   Потери на валидации: 3.6440994262695314
Эп 1 (Шаг 000000): Потери на обучении 2.633, Потери на валидации 2.622
...
Эп 2 (Шаг 000230): Потери на обучении 0.424, Потери на валидации 0.928
<|user|> Convert the active sentence to passive: 'The chef cooks the meal every day.' <|assistant|>: The meal is prepared every day by the chef....
Обучение завершено за 1.50 минут.
График сохранён как loss-plot-phi3-prompt.pdf
--------------------------------------------------
Генерация ответов
100% 110/110 [00:11<00:00,  9.27it/s]
Ответы сохранены как instruction-data-with-response-phi3-prompt.json
Модель сохранена как gpt2-medium355M-sft-phi3-prompt.pth
```

Для сравнения вы можете запустить оригинальный код тонкой настройки из главы 7 с помощью `python exercise_experiments.py --exercise_solution baseline`. 

Обратите внимание, что на GPU Nvidia L4 приведённый выше код с использованием шаблона промптов Phi-3 выполняется за 1,5 минуты. Для сравнения, шаблон в стиле Alpaca выполняется за 1,80 минуты. Таким образом, шаблон Phi-3 примерно на 17% быстрее, поскольку он приводит к более коротким входным данным модели. 

Давайте посмотрим на некоторые ответы, чтобы убедиться, что они отформатированы правильно:

```json
    {
        "instruction": "Rewrite the sentence using a simile.",
        "input": "The car is very fast.",
        "output": "The car is as fast as lightning.",
        "model_response": "The car is as fast as a cheetah."
    },
    {
        "instruction": "What type of cloud is typically associated with thunderstorms?",
        "input": "",
        "output": "The type of cloud typically associated with thunderstorms is cumulonimbus.",
        "model_response": "The type of cloud associated with thunderstorms is a cumulus cloud."
    },
    {
        "instruction": "Name the author of 'Pride and Prejudice'.",
        "input": "",
        "output": "Jane Austen.",
        "model_response": "The author of 'Pride and Prejudice' is Jane Austen."
    },
```

Мы можем оценить производительность с помощью метода Ollama Llama 3, который для вашего удобства также реализован в скрипте `python exercise_experiments.py` и который можно запустить следующим образом:

```bash
python ollama_evaluate.py --file_path instruction-data-with-response-phi3-prompt.json
```

Вывод:

```
Ollama запущен: True
Подсчёт оценок: 100%|████████████████████████| 110/110 [01:08<00:00,  1.60it/s]
Количество оценок: 110 из 110
Средняя оценка: 48.87
```

Оценка близка к 50, что находится в том же диапазоне, что и оценка, которую мы ранее получили с промптами в стиле Alpaca.

Нет никакого неотъемлемого преимущества или обоснования того, почему стиль промптов Phi должен быть лучше, но он может быть более кратким и эффективным, за исключением оговорки, упомянутой в разделе *Совет* ниже.

#### Совет: Учёт специальных токенов

- Обратите внимание, что шаблон промптов Phi-3 содержит специальные токены, такие как `<|user|>` и `<|assistant|>`, что может быть неоптимальным для токенизатора GPT-2
- Хотя токенизатор GPT-2 распознаёт `<|endoftext|>` как специальный токен (кодируемый в идентификатор токена 50256), он неэффективно обрабатывает другие специальные токены, такие как вышеупомянутые
- Например, `<|user|>` кодируется в 5 отдельных идентификаторов токенов (27, 91, 7220, 91, 29), что очень неэффективно
- Мы могли бы добавить `<|user|>` как новый специальный токен в `tiktoken` через аргумент `allowed_special`, но имейте в виду, что словарь GPT-2 не сможет обработать его без дополнительных модификаций
- Если вам интересно, как токенизатор и LLM могут быть расширены для обработки специальных токенов, пожалуйста, ознакомьтесь с дополнительными материалами [extend-tiktoken.ipynb](../../ch05/09_extending-tokenizers/extend-tiktoken.ipynb) (обратите внимание, что это не обязательно здесь, но представляет собой интересное/бонусное соображение для любознательных читателей)
- Кроме того, мы можем предположить, что модели, которые поддерживают эти специальные токены шаблона промптов через свой словарь, могут работать более эффективно и лучше в целом

---

- Мы используем форматирование промптов в стиле Alpaca, которое было исходным шаблоном промптов для инструктивной тонкой настройки
- Ниже мы форматируем входные данные, которые будем передавать в LLM

In [10]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

    return instruction_text + input_text

- Отформатированный ответ с полем ввода выглядит так, как показано ниже

In [11]:
model_input = format_input(data[50])
desired_response = f"\n\n### Response:\n{data[50]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Identify the correct spelling of the following word.

### Input:
Ocassion

### Response:
The correct spelling is 'Occasion.'


- Ниже представлен отформатированный ответ без поля ввода

In [12]:
model_input = format_input(data[999])
desired_response = f"\n\n### Response:\n{data[999]['output']}"

print(model_input + desired_response)

Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is an antonym of 'complicated'?

### Response:
An antonym of 'complicated' is 'simple'.


- Наконец, прежде чем мы подготовим загрузчики данных PyTorch в следующем разделе, мы разделим набор данных на обучающую, валидационную и тестовую выборки

In [13]:
train_portion = int(len(data) * 0.85)  # 85% на обучение
test_portion = int(len(data) * 0.1)    # 10% на тестирование
val_portion = len(data) - train_portion - test_portion  # Оставшиеся 5% на валидацию

train_data = data[:train_portion]
test_data = data[train_portion:train_portion + test_portion]
val_data = data[train_portion + test_portion:]

In [14]:
print("Длина обучающего набора:", len(train_data))
print("Длина валидационного набора:", len(val_data))
print("Длина тестового набора:", len(test_data))

Длина обучающего набора: 935
Длина валидационного набора: 55
Длина тестового набора: 110
